<a href="https://colab.research.google.com/github/kimheeseo/LSCNS/blob/main/2026_KICS_Fall/Carena/Carena_Fig5_GN_vs_ApproxEGN_nonexpert_report_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Carena Fig. 5 검증 보고서: main.py GN vs Approximate EGN

## 목적

Carena et al.의 2012년 Journal of Lightwave Technology 논문 Figure 5의 최대 전송거리 기준점과 비교하여, 기존 main.py의 GN 모델과 main_approx_egn.py의 Closed-form Approximate EGN 모델 중 어느 쪽이 더 가까운지 확인합니다.

이 보고서는 단순히 그래프 모양을 맞추는 작업이 아닙니다. 동일한 광섬유, WDM, span, launch-power 조건을 두 모델에 넣고 계산한 뒤, 마지막에만 논문 Figure 5의 digitized 기준점과 오차를 계산합니다.

## 핵심 용어

- GN: 신호를 가우시안 잡음처럼 근사해 비선형 잡음을 계산하는 빠른 해석 모델입니다.
- Approximate EGN: GN이 비선형 잡음을 다소 크게 계산할 수 있는 부분을 보정하는 닫힌 형태의 보정식입니다.
- 최대 전송거리: 요구 SNR을 만족하는 가장 긴 거리입니다.
- MAPE: 논문 거리와 계산 거리의 평균 절대 백분율 오차입니다. 작을수록 논문 기준점에 가깝습니다.

## 비교 범위와 공정성 규칙

1. GN과 Approximate EGN에는 같은 광섬유 입력, 9채널 WDM, 32 GBaud, 100 km span, NF 5 dB, launch-power grid를 사용합니다.
2. EGN 보정식에는 변조별 Phi 값이 필요합니다. 첨부된 closed-form EGN 논문에 값이 명시된 PM-QPSK와 PM-16QAM만 GN 대 EGN의 공통 비교 대상으로 사용합니다.
3. PM-BPSK와 PM-8QAM은 기존 GN Figure 5 보고서에는 포함되지만, 이 노트북의 EGN 승패 비교에는 임의의 Phi 값을 넣지 않기 위해 제외합니다.
4. Figure 5 점은 논문 원시 시뮬레이션 데이터가 아니라 그래프에서 digitize하고 반올림한 값입니다.
5. 어떤 논문값도 모델 계수, 입력값, launch power를 맞추는 데 사용하지 않습니다.

따라서 이 노트북의 결론은 "Carena Figure 5의 PM-QPSK 및 PM-16QAM 공통 30개 기준점에서 어떤 모델이 더 가까운가"입니다. 전체 EGN 적분 해석기 성능의 일반적 우열을 뜻하지는 않습니다.

In [ ]:
# Colab에서는 고정된 GitHub commit의 검증 코드를 내려받습니다.
# 로컬에서는 같은 폴더에 이미 있는 파일을 사용하므로 테스트가 가능합니다.
from pathlib import Path
from urllib.request import urlretrieve
import sys

REPOSITORY = "kimheeseo/LSCNS"
MAIN_PY_COMMIT = "7acd0ca1379592b6ad79acbe2748da4309830083"
APPROX_EGN_COMMIT = "48095432e3570e8af71733ca5343619b57ddb269"
SOURCE_PATH = "2026_KICS_Fall"
RUNNING_IN_COLAB = "google.colab" in sys.modules

required_sources = {
    "main.py": MAIN_PY_COMMIT,
    "main_approx_egn.py": APPROX_EGN_COMMIT,
}
for filename, commit in required_sources.items():
    local_file = Path(filename)
    if RUNNING_IN_COLAB or not local_file.exists():
        url = (
            f"https://raw.githubusercontent.com/{REPOSITORY}/{commit}/"
            f"{SOURCE_PATH}/{filename}"
        )
        urlretrieve(url, local_file)
        print("Downloaded:", url)
    else:
        print("Using local source:", local_file.resolve())

for module_name in ("main", "main_approx_egn"):
    sys.modules.pop(module_name, None)
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:
    from IPython.display import Markdown, display
except ImportError:
    class Markdown(str):
        pass

    def display(value):
        print(value)

from main import (
    FIG5_BITS_PER_SYMBOL,
    FIG5_FIBER_INPUTS,
    FIG5_REFERENCE_DOI,
    FIG5_REFERENCE_REACH_KM,
    FIG5_REQUIRED_OSNR_DB_PER_0P1NM,
    FiberParameters,
    GNOptions,
    SystemParameters,
    _maximum_reach_from_gn,
    derive_parameters,
    gn_eta_per_span,
)
from main_approx_egn import (
    ApproxEGNOptions,
    PAPER_EQUATIONS,
    PAPER_REFERENCE,
    approximate_egn_eta_per_span,
    modulation_phi,
)

pd.set_option("display.max_rows", 200)
print("GN source:", MAIN_PY_COMMIT)
print("Approximate EGN source:", APPROX_EGN_COMMIT)

## 1. 논문 Figure 5에 입력한 조건

광섬유별로 손실, 분산, 비선형계수 gamma만 바꾸고 나머지 WDM 조건은 동일하게 둡니다. 이 조건은 기존 Carena Figure 5 GN 보고서와 같으므로, GN과 Approximate EGN의 차이는 EGN 보정항에서만 발생합니다.

In [ ]:
common_inputs = pd.DataFrame(
    [
        ["채널 수", "9", "동시에 전송하는 WDM 채널 수"],
        ["Symbol rate", "32 GBaud", "한 채널의 신호 속도"],
        ["Span 길이", "100 km", "증폭기 사이의 광섬유 길이"],
        ["증폭기 NF", "5 dB", "증폭기가 추가하는 잡음 수준"],
        ["ASE 대역폭", "32 GHz", "기존 main.py Figure 5 조건"],
        ["Launch-power grid", "-10 ~ +8 dBm, 0.05 dB step", "두 모델이 공통으로 탐색"],
        ["최대 span 수", "300", "최대 30,000 km까지 탐색"],
        ["NLI 누적", "N span", "기존 Figure 5 비교와 동일한 비간섭 누적"],
        ["GN 계수", "8/27", "main.py의 total-DP power 정의를 유지"],
    ],
    columns=["입력 항목", "값", "의미"],
)
fiber_inputs = (
    pd.DataFrame(FIG5_FIBER_INPUTS)
    .T.rename(
        columns={
            "attenuation_db_per_km": "손실 (dB/km)",
            "dispersion_ps_nm_km": "분산 (ps/nm/km)",
            "gamma_per_w_km": "비선형계수 gamma (1/W/km)",
        }
    )
    .rename_axis("광섬유")
    .reset_index()
)
egn_modulations = pd.DataFrame(
    [
        ["PM-QPSK", modulation_phi("PM-QPSK"), 6],
        ["PM-16QAM", modulation_phi("PM-16QAM"), 4],
    ],
    columns=["공통 비교 변조방식", "EGN Phi", "광섬유당 Figure 5 점 수"],
)

display(common_inputs)
display(fiber_inputs)
display(egn_modulations)

## 2. 계산 흐름

1. Figure 5의 OSNR 요구값을 32 GBaud 채널의 SNR 기준으로 변환합니다.
2. GN eta를 계산합니다.
3. 같은 GN eta에서 Approximate EGN 보정 eta를 빼서 EGN eta를 계산합니다.
4. 각 launch power와 span 수에서 SNR을 계산하고, 기준 SNR을 만족하는 최대 span을 찾습니다.
5. 논문 Figure 5 거리와 비교하여 GN 및 EGN의 오차율을 계산합니다.

논문 기준점은 마지막 단계에서만 사용됩니다. 즉, 이 노트북에는 eta fitting, launch-power fitting, 오차 최소화 최적화가 없습니다.

In [ ]:
COMPARISON_MODULATIONS = ("PM-QPSK", "PM-16QAM")
LAUNCH_DBM = np.arange(-10.0, 8.0 + 0.001, 0.05)
MAX_SPANS = 300
GN_OPTIONS = GNOptions(
    nli_coefficient=8.0 / 27.0,
    finite_effective_length=True,
    power_definition="total_dp",
    model_name="gn_finite_main_py_baseline",
)


def make_figure5_system(spacing_ghz):
    return SystemParameters(
        channels=9,
        symbol_rate_gbd=32.0,
        spacing_ghz=float(spacing_ghz),
        span_length_km=100.0,
        noise_figure_db=5.0,
        transceiver_snr_db=18.0,
        polarizations=2,
        ase_bandwidth_hz=32e9,
        stated_gain_bandwidth_thz=None,
    )


def osnr_to_required_snr_db(required_osnr_db):
    return float(required_osnr_db + 10.0 * np.log10(12.5e9 / 32e9))


def maximum_reach(eta_per_span, parameters, system, required_snr_db):
    return _maximum_reach_from_gn(
        LAUNCH_DBM,
        eta_per_span=float(eta_per_span),
        parameters=parameters,
        system=system,
        required_snr_db=float(required_snr_db),
        max_spans=MAX_SPANS,
        accumulation_exponent=1.0,
        include_transceiver_noise=False,
    )


def run_gn_vs_approximate_egn():
    rows = []
    for fiber_name, fiber_input in FIG5_FIBER_INPUTS.items():
        fiber = FiberParameters(name=fiber_name, **fiber_input)
        for modulation in COMPARISON_MODULATIONS:
            for spacing_ghz, required_osnr_db in (
                FIG5_REQUIRED_OSNR_DB_PER_0P1NM[modulation].items()
            ):
                system = make_figure5_system(spacing_ghz)
                parameters = derive_parameters(fiber, system)
                eta_gn = gn_eta_per_span(parameters, system, GN_OPTIONS)
                egn_info = approximate_egn_eta_per_span(
                    fiber=fiber,
                    system=system,
                    gn_options=GN_OPTIONS,
                    egn_options=ApproxEGNOptions(
                        modulation_format=modulation,
                        strict_applicability=True,
                    ),
                )
                eta_egn = egn_info["eta_approx_egn_per_span_w_inv2"]
                required_snr_db = osnr_to_required_snr_db(required_osnr_db)
                paper_reach_km = float(
                    FIG5_REFERENCE_REACH_KM[fiber_name][modulation][
                        spacing_ghz
                    ]
                )
                for model_name, eta in (
                    ("GN", eta_gn),
                    ("Approximate EGN", eta_egn),
                ):
                    reach = maximum_reach(
                        eta, parameters, system, required_snr_db
                    )
                    predicted_reach_km = float(reach["max_reach_km"])
                    error_pct = 100.0 * (
                        predicted_reach_km - paper_reach_km
                    ) / paper_reach_km
                    rows.append(
                        {
                            "fiber": fiber_name,
                            "modulation": modulation,
                            "spacing_ghz": float(spacing_ghz),
                            "net_spectral_efficiency_bps_hz": float(
                                FIG5_BITS_PER_SYMBOL[modulation]
                                * 25.0
                                / spacing_ghz
                            ),
                            "model": model_name,
                            "required_osnr_db_per_0p1nm": float(
                                required_osnr_db
                            ),
                            "required_snr_db": required_snr_db,
                            "paper_reach_km": paper_reach_km,
                            "predicted_reach_km": predicted_reach_km,
                            "launch_dbm_at_max_reach": float(
                                reach["launch_dbm_at_max_reach"]
                            ),
                            "eta_per_span_w_inv2": float(eta),
                            "egn_eta_reduction_pct": float(
                                egn_info["nli_eta_reduction_percent"]
                            ),
                            "error_pct": float(error_pct),
                            "abs_error_pct": float(abs(error_pct)),
                            "reach_error_db": float(
                                10.0
                                * np.log10(
                                    predicted_reach_km / paper_reach_km
                                )
                            ),
                        }
                    )
    return pd.DataFrame(rows).sort_values(
        ["fiber", "modulation", "spacing_ghz", "model"]
    ).reset_index(drop=True)


def summarize_errors(frame, group_columns):
    group_columns = list(group_columns) + ["model"]
    records = []
    for keys, group in frame.groupby(group_columns, sort=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        record = dict(zip(group_columns, keys))
        record.update(
            {
                "points": int(len(group)),
                "mape_pct": float(group["abs_error_pct"].mean()),
                "bias_pct": float(group["error_pct"].mean()),
                "reach_rmse_db": float(
                    np.sqrt(np.mean(group["reach_error_db"] ** 2))
                ),
                "within_10pct_share": float(
                    100.0 * np.mean(group["abs_error_pct"] <= 10.0)
                ),
            }
        )
        records.append(record)
    return pd.DataFrame(records)

In [ ]:
comparison = run_gn_vs_approximate_egn()
overall_summary = summarize_errors(comparison, [])
fiber_summary = summarize_errors(comparison, ["fiber"])
modulation_summary = summarize_errors(comparison, ["modulation"])

print("공통 비교 기준점 수:", comparison[["fiber", "modulation", "spacing_ghz"]].drop_duplicates().shape[0])
display(overall_summary.round(3))
display(fiber_summary.round(3))
display(modulation_summary.round(3))

## 3. 논문 Figure 5 값과 GN 및 Approximate EGN 계산값

각 그래프에서 원형 실선은 논문 Figure 5에서 digitize한 기준점이고, X 점선은 GN, 삼각형 점선은 Approximate EGN 계산값입니다. 세로축은 최대 전송거리이므로 로그 눈금을 사용합니다.

In [ ]:
model_styles = {
    "GN": {"marker": "x", "linestyle": "--", "label": "GN"},
    "Approximate EGN": {
        "marker": "^",
        "linestyle": ":",
        "label": "Approximate EGN",
    },
}
modulation_colors = {
    "PM-QPSK": "#ff7f0e",
    "PM-16QAM": "#d62728",
}

figure_reach, axes = plt.subplots(
    1, 3, figsize=(16, 5.2), sharey=True, layout="constrained"
)
for axis, fiber_name in zip(axes, FIG5_FIBER_INPUTS):
    fiber_frame = comparison.loc[comparison["fiber"] == fiber_name]
    for modulation in COMPARISON_MODULATIONS:
        modulation_frame = fiber_frame.loc[
            fiber_frame["modulation"] == modulation
        ].sort_values("net_spectral_efficiency_bps_hz")
        paper = modulation_frame.loc[
            modulation_frame["model"] == "GN"
        ]
        color = modulation_colors[modulation]
        axis.plot(
            paper["net_spectral_efficiency_bps_hz"],
            paper["paper_reach_km"],
            "o-",
            color=color,
            lw=1.8,
            ms=4,
            label=f"{modulation} paper",
        )
        for model_name, style in model_styles.items():
            predicted = modulation_frame.loc[
                modulation_frame["model"] == model_name
            ]
            axis.plot(
                predicted["net_spectral_efficiency_bps_hz"],
                predicted["predicted_reach_km"],
                color=color,
                lw=1.5,
                ms=5,
                marker=style["marker"],
                linestyle=style["linestyle"],
                label=f"{modulation} {style['label']}",
            )
    axis.set_title(fiber_name)
    axis.set_xlabel("Net spectral efficiency (bit/s/Hz)")
    axis.set_yscale("log")
    axis.grid(alpha=0.25, which="both")
axes[0].set_ylabel("Maximum reach (km)")
axes[0].legend(fontsize=7, ncol=2, loc="best")
figure_reach.suptitle(
    "Carena et al. Figure 5: digitized paper values vs GN and Approximate EGN",
    y=1.02,
)
plt.show()

## 4. 어느 모델이 논문값에 더 가까운가

아래 MAPE는 같은 30개 공통 기준점에서 계산한 값입니다. 낮은 막대가 이번 검증 조건에서 Figure 5에 더 가깝다는 뜻입니다.

In [ ]:
ordered_models = ["GN", "Approximate EGN"]
mape_table = (
    overall_summary.set_index("model")
    .loc[ordered_models, "mape_pct"]
    .rename("MAPE (%)")
)
figure_mape, axis = plt.subplots(figsize=(6.8, 4.2), layout="constrained")
bars = axis.bar(
    mape_table.index,
    mape_table.values,
    color=["#1f77b4", "#9467bd"],
)
axis.set_ylabel("Mean absolute percentage error (%)")
axis.set_title("Carena Figure 5 common subset: lower is better")
axis.grid(axis="y", alpha=0.3)
for bar, value in zip(bars, mape_table.values):
    axis.text(
        bar.get_x() + bar.get_width() / 2,
        value,
        f"{value:.2f}%",
        ha="center",
        va="bottom",
    )
plt.show()

fiber_mape = fiber_summary.pivot(
    index="fiber", columns="model", values="mape_pct"
).reindex(columns=ordered_models)
display(fiber_mape.round(3))

gn_rmse_db = float(
    overall_summary.set_index("model").loc["GN", "reach_rmse_db"]
)
egn_rmse_db = float(
    overall_summary.set_index("model").loc[
        "Approximate EGN", "reach_rmse_db"
    ]
)
display(
    Markdown(
        "MAPE를 모델 우열의 주 지표로 사용합니다. "
        f"참고로 log-distance RMSE는 GN {gn_rmse_db:.3f} dB, "
        f"Approximate EGN {egn_rmse_db:.3f} dB입니다. "
        "MAPE와 log-RMSE의 순위가 다를 수 있는데, 두 지표가 "
        "짧은 거리와 긴 거리를 가중하는 방식이 다르기 때문입니다."
    )
)

In [ ]:
overall_by_model = overall_summary.set_index("model")
gn_mape = float(overall_by_model.loc["GN", "mape_pct"])
egn_mape = float(overall_by_model.loc["Approximate EGN", "mape_pct"])
winner = "GN" if gn_mape < egn_mape else "Approximate EGN"
relative_change_pct = 100.0 * (egn_mape - gn_mape) / gn_mape

paired = comparison.pivot_table(
    index=["fiber", "modulation", "spacing_ghz"],
    columns="model",
    values=["abs_error_pct", "error_pct", "paper_reach_km", "predicted_reach_km"],
)
per_case_winner = np.where(
    paired["abs_error_pct"]["GN"] <= paired["abs_error_pct"]["Approximate EGN"],
    "GN",
    "Approximate EGN",
)
gn_win_count = int(np.sum(per_case_winner == "GN"))
egn_win_count = int(np.sum(per_case_winner == "Approximate EGN"))

if winner == "GN":
    conclusion = (
        f"이번 공통 30개 기준점에서는 GN의 MAPE가 {gn_mape:.2f}%로 "
        f"Approximate EGN의 {egn_mape:.2f}%보다 낮았습니다. "
        f"현재 main.py 조건에서는 EGN 보정 후 MAPE가 {relative_change_pct:.1f}% 커졌습니다."
    )
else:
    conclusion = (
        f"이번 공통 30개 기준점에서는 Approximate EGN의 MAPE가 {egn_mape:.2f}%로 "
        f"GN의 {gn_mape:.2f}%보다 낮았습니다. "
        f"EGN 보정으로 MAPE가 {-relative_change_pct:.1f}% 줄었습니다."
    )

display(Markdown("### 자동 계산 결론"))
display(Markdown(conclusion))
print(f"점별 승리 횟수: GN {gn_win_count}개, Approximate EGN {egn_win_count}개")

## 5. 오차가 생기는 이유

이번 결과에서 GN과 Approximate EGN의 오차가 같지 않은 이유는 다음과 같습니다.

1. Approximate EGN은 GN보다 NLI를 줄이는 방향의 보정입니다. 따라서 EGN은 최대 전송거리를 GN보다 길게 예측합니다. GN이 이미 논문값보다 짧은 거리를 예측한 경우에는 도움이 될 수 있지만, GN이 논문값에 가깝거나 길게 예측한 경우에는 오히려 오차가 커질 수 있습니다.
2. 이 EGN 파일은 full EGN 적분 해석기가 아니라 closed-form Eq. (1)과 Eq. (3) 보정식입니다. 동일 채널, 동일 간격, 중앙 채널, 이상적 사각 스펙트럼, lumped EDFA, 충분한 span 수를 가정합니다.
3. main.py의 GN 기준식은 Figure 5 비교를 위해 사용한 중앙 채널 closed-form 근사입니다. EGN 보정식이 유도된 coherent GN PSD와 전력 정의, pulse shape, NLI 누적 모델이 완전히 일치하지 않으면 보정량이 과대 또는 과소가 될 수 있습니다.
4. Figure 5 기준점은 digitize 및 반올림 값이고, 본 코드는 100 km 단위 span으로 최대거리를 고릅니다. 짧은 거리에서는 한 span 차이가 큰 백분율 오차가 됩니다.
5. OSNR 0.1 nm 기준을 32 GHz SNR 기준으로 변환하는 과정과 실제 수신기 filtering 또는 pulse roll-off 차이도 잔여 오차를 만들 수 있습니다.

따라서 여기서 EGN이 GN보다 MAPE가 크더라도 "EGN 이론이 나쁘다"는 뜻은 아닙니다. 현재 main.py의 GN 식과 이 논문의 closed-form 보정식을 결합한 구현이 Carena Figure 5의 digitized 기준점에 얼마나 맞는지에 대한 결과입니다.

In [ ]:
gn_bias = float(overall_by_model.loc["GN", "bias_pct"])
egn_bias = float(overall_by_model.loc["Approximate EGN", "bias_pct"])
eta_reduction = (
    comparison.loc[comparison["model"] == "Approximate EGN", "egn_eta_reduction_pct"]
    .mean()
)

print("오차 방향 확인")
print(f"- GN 평균 bias: {gn_bias:+.2f}%")
print(f"- Approximate EGN 평균 bias: {egn_bias:+.2f}%")
print(f"- Approximate EGN의 평균 NLI eta 감소량: {eta_reduction:.2f}%")
print(
    "- 양의 bias는 계산거리가 논문 기준보다 길다는 뜻이고, "
    "음의 bias는 더 짧다는 뜻입니다."
)

## 6. 점별 오차 확인

아래 표는 절대 오차가 큰 조건부터 보여 줍니다. 논문값이 짧은 조건에서는 100 km 한 span 차이가 큰 퍼센트로 보일 수 있으므로, 오차율과 함께 km 차이도 같이 봐야 합니다.

In [ ]:
display_columns = [
    "fiber",
    "modulation",
    "spacing_ghz",
    "model",
    "paper_reach_km",
    "predicted_reach_km",
    "launch_dbm_at_max_reach",
    "error_pct",
    "abs_error_pct",
]
worst_cases = (
    comparison.sort_values("abs_error_pct", ascending=False)
    .head(12)[display_columns]
    .reset_index(drop=True)
)
display(worst_cases.round(3))

## 7. 결과 저장

아래 셀은 비교 표, 요약 표, 실행 조건 JSON, 두 그래프를 carena_fig5_gn_vs_approx_egn_results 폴더에 저장합니다. Colab 왼쪽 파일 창에서 내려받을 수 있습니다.

In [ ]:
output_directory = Path("carena_fig5_gn_vs_approx_egn_results")
output_directory.mkdir(exist_ok=True)

comparison.to_csv(
    output_directory / "carena_fig5_gn_vs_approx_egn_comparison.csv",
    index=False,
    encoding="utf-8-sig",
)
overall_summary.to_csv(
    output_directory / "carena_fig5_gn_vs_approx_egn_overall_summary.csv",
    index=False,
    encoding="utf-8-sig",
)
fiber_summary.to_csv(
    output_directory / "carena_fig5_gn_vs_approx_egn_fiber_summary.csv",
    index=False,
    encoding="utf-8-sig",
)
figure_reach.savefig(
    output_directory / "carena_fig5_gn_vs_approx_egn_reach.png",
    dpi=180,
    bbox_inches="tight",
)
figure_mape.savefig(
    output_directory / "carena_fig5_gn_vs_approx_egn_mape.png",
    dpi=180,
    bbox_inches="tight",
)

metadata = {
    "carena_paper": {
        "citation": (
            "A. Carena et al., Modeling of the Impact of Nonlinear "
            "Propagation Effects in Uncompensated Optical Coherent "
            "Transmission Links, JLT 30(10), 1524-1539, 2012."
        ),
        "doi": FIG5_REFERENCE_DOI,
        "reference_data": "Digitized and rounded Figure 5 markers",
    },
    "approximate_egn_paper": {
        "citation": PAPER_REFERENCE,
        "equations": PAPER_EQUATIONS,
    },
    "source_commits": {
        "main_py": MAIN_PY_COMMIT,
        "main_approx_egn_py": APPROX_EGN_COMMIT,
    },
    "comparison_scope": {
        "modulations": list(COMPARISON_MODULATIONS),
        "points": int(
            comparison[["fiber", "modulation", "spacing_ghz"]]
            .drop_duplicates()
            .shape[0]
        ),
        "no_fitting": True,
    },
    "overall_metrics": overall_summary.to_dict(orient="records"),
}
with (output_directory / "carena_fig5_gn_vs_approx_egn_metadata.json").open(
    "w", encoding="utf-8"
) as file:
    json.dump(metadata, file, indent=2, ensure_ascii=False)

print("저장 위치:", output_directory.resolve())
print("생성 파일:", [path.name for path in sorted(output_directory.iterdir())])

## 참고문헌

1. A. Carena, V. Curri, G. Bosco, P. Poggiolini, and F. Forghieri, "Modeling of the Impact of Nonlinear Propagation Effects in Uncompensated Optical Coherent Transmission Links," Journal of Lightwave Technology, vol. 30, no. 10, pp. 1524-1539, 2012. DOI: 10.1109/JLT.2012.2189198.
2. P. Poggiolini et al., "A Simple and Accurate Closed-Form EGN Model Formula," arXiv:1503.04132v1, 2015.

주의: 이 노트북은 두 논문의 원시 시뮬레이션 데이터나 full EGN 적분 코드를 재현한 것이 아닙니다. main.py GN 엔진과 main_approx_egn.py의 closed-form Approximate EGN 보정식을 같은 Figure 5 digitized 기준점에 비교하는 재현성 보고서입니다.